In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [3]:
import kagglehub

In [4]:
import kagglehub

path = kagglehub.dataset_download(
    "tongpython/cat-and-dog"
)

print("Dataset path:", path)

Using Colab cache for faster access to the 'cat-and-dog' dataset.
Dataset path: /kaggle/input/cat-and-dog


In [5]:
import os

print(os.listdir(path))

['test_set', 'training_set']


In [6]:
print(os.listdir(path))

['test_set', 'training_set']


In [7]:
import os

print("Dataset path:", path)
print(os.listdir(path))

Dataset path: /kaggle/input/cat-and-dog
['test_set', 'training_set']


In [8]:
for item in os.listdir(path):
    item_path = os.path.join(path, item)
    if os.path.isdir(item_path):
        print(item, "->", os.listdir(item_path)[:5])

test_set -> ['test_set']
training_set -> ['training_set']


In [9]:
import os

train_path = os.path.join(path, "training_set", "training_set")
test_path = os.path.join(path, "test_set", "test_set")

print("Training folders:", os.listdir(train_path))
print("Test folders:", os.listdir(test_path))

Training folders: ['dogs', 'cats']
Test folders: ['dogs', 'cats']


In [10]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [11]:
train_dataset = datasets.ImageFolder(
    train_path,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    test_path,
    transform=transform
)

print("Train Images:", len(train_dataset))
print("Test Images:", len(test_dataset))
print("Classes:", train_dataset.classes)

Train Images: 8005
Test Images: 2023
Classes: ['cats', 'dogs']


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("DataLoaders ready!")

DataLoaders ready!


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 142MB/s]


Linear(in_features=512, out_features=2, bias=True)


In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Ready for training!")

Ready for training!


In [15]:
epochs = 3

for epoch in range(epochs):

    running_loss = 0.0

    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/3, Loss: 0.1608
Epoch 2/3, Loss: 0.0971
Epoch 3/3, Loss: 0.0809


In [16]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 97.58%
